# Chapter 3: Protein Identification and Quantification (Open-Source DIA Workflow)

### 3.1 Environment Verification


To ensure reproducibility of the proteomics workflow, we first confirm the operating system and Linux distribution used in this analysis. Google Colab provides an Ubuntu-based Linux environment suitable for open-source proteomics tools.

In [ ]:
!uname -a
!lsb_release -a


Linux 828507e92e92 6.6.105+ #1 SMP Thu Oct  2 10:42:05 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy


### 3.2 System Dependencies Installation



Core system utilities required for downloading data, compiling software, and running proteomics tools were installed using the Ubuntu package manager.

In [ ]:
!apt-get update
!apt-get install -y wget unzip git build-essential


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,573 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,966 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Get:14 ht

In [ ]:
!mkdir data

In [ ]:
!mkdir -p data

In [ ]:
%cd data

/content/data


### 3.3 Data Retrieval from ProteomeXchange
Publicly available DIA mass spectrometry data were retrieved from the ProteomeXchange Consortium (dataset ID: PXD058672). The corresponding mzML files for normal and tumor samples were downloaded directly from the ProteomeCentral repository.


In [ ]:
!wget -O CRC03-N.mzML \
"https://proteomecentral.proteomexchange.org/cgi/GetDataset?ID=PXD058672&file=CRC03-N.mzML"

!wget -O CRC03-T.mzML \
"https://proteomecentral.proteomexchange.org/cgi/GetDataset?ID=PXD058672&file=CRC03-T.mzML"


--2026-01-05 02:03:03--  https://proteomecentral.proteomexchange.org/cgi/GetDataset?ID=PXD058672&file=CRC03-N.mzML
Resolving proteomecentral.proteomexchange.org (proteomecentral.proteomexchange.org)... 174.127.185.143
Connecting to proteomecentral.proteomexchange.org (proteomecentral.proteomexchange.org)|174.127.185.143|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘CRC03-N.mzML’

CRC03-N.mzML            [ <=>                ]   6.38K  --.-KB/s    in 0.001s  

2026-01-05 02:03:04 (10.4 MB/s) - ‘CRC03-N.mzML’ saved [6528]

--2026-01-05 02:03:04--  https://proteomecentral.proteomexchange.org/cgi/GetDataset?ID=PXD058672&file=CRC03-T.mzML
Resolving proteomecentral.proteomexchange.org (proteomecentral.proteomexchange.org)... 174.127.185.143
Connecting to proteomecentral.proteomexchange.org (proteomecentral.proteomexchange.org)|174.127.185.143|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [

In [ ]:
!ls -lh

total 16K
-rw-r--r-- 1 root root 6.4K Jan  5 02:03 CRC03-N.mzML
-rw-r--r-- 1 root root 6.4K Jan  5 02:03 CRC03-T.mzML


### 3.4 Open-Source DIA Processing Strategy
To reproduce the analytical logic of DIA-NN using fully open-source tools, we implemented a DIA-based identification and quantification workflow using OpenMS. This workflow mirrors DIA-NN concepts including in silico spectral library generation, peptide-centric scoring, false discovery rate (FDR) control, and protein-level quantification.


### 3.5 Human Protein Sequence Database Preparation

Protein identification was performed using the human reference proteome obtained from UniProt (proteome ID: UP000005640). This FASTA database provides the protein sequences required for in silico digestion and spectral library generation, analogous to the FASTA-based library generation approach used in DIA-NN.


In [ ]:
!wget -O human_proteome.fasta \
"https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=proteome:UP000005640"


--2026-01-05 17:06:28--  https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=proteome:UP000005640
Resolving rest.uniprot.org (rest.uniprot.org)... 193.62.193.81
Connecting to rest.uniprot.org (rest.uniprot.org)|193.62.193.81|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [text/plain]
Saving to: ‘human_proteome.fasta’

human_proteome.fast     [   <=>              ]  38.77M   428KB/s    in 59s     

2026-01-05 17:07:29 (667 KB/s) - ‘human_proteome.fasta’ saved [40649282]



In [ ]:
!head -n 5 human_proteome.fasta
!grep -c ">" human_proteome.fasta


>tr|A0A087WVL8|A0A087WVL8_HUMAN Fragile X messenger ribonucleoprotein 1 OS=Homo sapiens OX=9606 GN=FMR1 PE=1 SV=1
MEELVVEVRGSNGAFYKAFVKDVHEDSITVAFENNWQPDRQIPFHDVRFPPPVGYNKDIN
ESDEVEVYSRANEKEPCCWWLAKVRMIKGEFYVIEYAACDATYNEIVTIERLRSVNPNKP
ATKDTFHKIKLDVPEDLRQMCAKEAAHKDFKKAVGAFSVTYDPENYQLVILSINEVTSKR
AHMLIDMHFRSLRTKLSLIMRNEEASKQLESSRQLASRFHEQFIVREDLMGLAIGTHGAN
83607


### 3.6 In-silico Protein Digestion

To reproduce DIA-NN’s FASTA-based spectral library generation, the human protein FASTA database was digested in silico using trypsin. One missed cleavage was allowed, and peptides were filtered based on length and charge state to match DIA acquisition constraints.


In [ ]:
!apt-get install -y openms

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core coinor-libcbc3 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5 gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0
  libboost-math1.74.0 libboost-regex1.74.0 libdouble-conversion3 libevdev2
  libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0 libinput-bin
  libinput10 libmd4c0 libmtdev1 libopenms2.6.0 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 librsvg2-common libsvm3
  libwacom-bin libwacom-common libwacom9 libwildmagic-common libwildmagic5
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 openms-common qt5-gtk-platformtheme
  qttranslations5-l10n session-migration topp
Suggested packages:
  gvfs openms-doc 

In [ ]:
!Digestor \
-in human_proteome.fasta \
-out human_tryptic_peptides.fasta \
-enzyme Trypsin \
-missed_cleavages 1 \
-min_length 7 \
-max_length 45


Statistics:
  file:                                    human_proteome.fasta
  total #peptides after digestion:         6791785
  removed #peptides (length restrictions): 2700680
  remaining #peptides:                     4091105
Digestor took 9.83 s (wall), 9.69 s (CPU), 0.22 s (system), 9.47 s (user); Peak Memory Usage: 96 MB.


In [ ]:
!head -n 5 human_tryptic_peptides.fasta
!grep -c ">" human_tryptic_peptides.fasta


>tr|A0A087WVL8|A0A087WVL8_HUMAN 
MEELVVEVR
>tr|A0A087WVL8|A0A087WVL8_HUMAN 
GSNGAFYK
>tr|A0A087WVL8|A0A087WVL8_HUMAN 
4091105


The human proteome FASTA database was successfully digested in silico using trypsin with one missed cleavage allowed. This resulted in millions of tryptic peptides, consistent with expectations for the full human proteome. Example tryptic peptides (ending in K or R) were verified, confirming correct enzymatic digestion and peptide-length filtering.


In [ ]:
!apt-get install -y openms


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core coinor-libcbc3 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5 gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0
  libboost-math1.74.0 libboost-regex1.74.0 libdouble-conversion3 libevdev2
  libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0 libinput-bin
  libinput10 libmd4c0 libmtdev1 libopenms2.6.0 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 librsvg2-common libsvm3
  libwacom-bin libwacom-common libwacom9 libwildmagic-common libwildmagic5
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 openms-common qt5-gtk-platformtheme
  qttranslations5-l10n session-migration topp
Suggested packages:
  gvfs openms-doc 